# DeepSetZ — Interactive Evaluation

Select a trained model and choose which filters to include.  
Press **Run** to evaluate on the test set and see live metrics and plots.

> **Setup:** Run the first code cell once to load the backend, then run the second cell to launch the UI.

In [1]:
# ── Setup — run this cell once ──────────────────────────────────────────────
# %matplotlib inline must be the effective backend before any other import
%matplotlib inline

import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display as ipy_display
from torch.utils.data import DataLoader

from src.config import load_config
from src.dataset import GalaxyDataset, collate_fn
from src.train import build_model
from src.evaluate import compute_metrics, OUTLIER_THRESHOLD
from src.run_artifacts import (
    run_is_complete, available_roles, checkpoint_path, role_label,
    load_post_hoc, post_hoc_sigma_scale,
    ROLE_POINT, ROLE_POSTERIOR, ROLE_CALIBRATED, ROLE_END_TO_END,
)
from src.calibration import apply_sigma_scale_to_out, pit_mdn, pit_nsf
from src.models.heads.nsf import NeuralSplineFlow
from src.models.heads.mdn        import MDN
from src.models.heads.binned_pdf import BinnedPDF
from src.models.heads.nsf        import NeuralSplineFlow

PROB_HEADS = (MDN, BinnedPDF, NeuralSplineFlow)
from src.benchmarks.models import FlatBenchmarkMLP, FlatBenchmarkMDN
from src.benchmarks.dataset import FlatVectorDataset
from src.benchmarks.train import build_benchmark_model

# ── Human-readable filter names ─────────────────────────────────────────────
# Short names where unambiguous; survey-prefixed where bands overlap.
FILTER_LABELS = {
    # Ellen / noiseless
    "mag_u_lsst":   "u",
    "mag_g_lsst":   "g",
    "mag_r_lsst":   "r",
    "mag_i_lsst":   "i",
    "mag_z_lsst":   "z",
    "mag_y_lsst":   "y",
    "Roman_Y106":   "Roman Y",
    "Roman_J129":   "Roman J",
    "Roman_H158":   "Roman H",
    "Roman_F184":   "Roman F",
    "Roman_K213":   "Roman K",
    "Euclid_Y":     "Euclid Y",
    "Euclid_J":     "Euclid J",
    "Euclid_H":     "Euclid H",
    "WISE_W1":      "W1",
    "WISE_W2":      "W2",
    # PZDC column names (letter-band style)
    "mag_Y_roman":  "Roman Y",
    "mag_J_roman":  "Roman J",
    "mag_H_roman":  "Roman H",
    # PZDC column names (filter-number style)
    "mag_F106_roman": "Roman Y",
    "mag_F129_roman": "Roman J",
    "mag_F158_roman": "Roman H",
    "mag_F184_roman": "Roman F",
    "mag_F213_roman": "Roman K",
}

# Survey grouping order for the UI
SURVEY_ORDER = ["lsst", "roman", "euclid", "wise"]
# All known column names for each survey across both datasets.
# Ellen uses capitalised names (Roman_Y106); PZDC uses snake_case (mag_F106_roman).
# rebuild_filters intersects this list with the model's actual active columns,
# so listing both naming conventions here is safe — unused ones are ignored.
SURVEY_DISPLAY = {
    "lsst":   ["mag_u_lsst", "mag_g_lsst", "mag_r_lsst",
               "mag_i_lsst", "mag_z_lsst", "mag_y_lsst"],
    "roman":  [
        # Ellen naming
        "Roman_Y106", "Roman_J129", "Roman_H158", "Roman_F184", "Roman_K213",
        # PZDC letter-band naming
        "mag_Y_roman", "mag_J_roman", "mag_H_roman",
        # PZDC filter-number naming
        "mag_F106_roman", "mag_F129_roman", "mag_F158_roman",
        "mag_F184_roman", "mag_F213_roman",
    ],
    "euclid": ["Euclid_Y",   "Euclid_J",   "Euclid_H"],
    "wise":   ["WISE_W1",    "WISE_W2"],
}
SURVEY_TITLES = {"lsst": "LSST", "roman": "Roman", "euclid": "Euclid", "wise": "WISE"}

# ── Model discovery ──────────────────────────────────────────────────────────
def find_runs(outputs_dir: Path, benchmarks_dir: Path | None = None) -> dict[str, Path]:
    """Discover DeepSetZ runs (outputs/) and flat benchmarks (benchmarks/)."""
    runs: dict[str, Path] = {}
    for base, prefix in ((outputs_dir, ""), (benchmarks_dir, "[bench] ")):
        if base is None or not base.exists():
            continue
        for d in sorted(base.iterdir()):
            if d.is_dir() and run_is_complete(d):
                label = f"{prefix}{d.name}" if prefix else d.name
                runs[label] = d
    return runs

def _cache_key(run_name: str, role: str) -> str:
    return f"{run_name}::{role}"

def default_role(run_dir: Path) -> str:
    roles = available_roles(run_dir)
    if ROLE_POSTERIOR in roles:
        return ROLE_POSTERIOR
    if ROLE_END_TO_END in roles:
        return ROLE_END_TO_END
    if ROLE_POINT in roles:
        return ROLE_POINT
    return ROLE_END_TO_END

OUTPUTS_DIR    = ROOT / "outputs"
BENCHMARKS_DIR = ROOT / "benchmarks"

def _resolve_run_dir(run_name: str) -> Path:
    if run_name.startswith("[bench] "):
        return BENCHMARKS_DIR / run_name[len("[bench] "):]
    return OUTPUTS_DIR / run_name

# ── Model + data cache (avoid reloading on every Run) ───────────────────────
_cache:      dict = {}   # run_name → state dict
_prob_cache: dict = {}   # (run_name, cols_key) → {z_mean, z_median, z_mode, pit}

def _load_benchmark_run(run_name: str, run_dir: Path, cfg) -> dict:
    """Load a flat-vector benchmark (fixed filter subset, no filter picker)."""
    filter_cols = cfg.benchmark.filter_columns
    test_path   = ROOT / cfg.data.test_path

    train_path = ROOT / cfg.data.train_path
    train_ds = FlatVectorDataset(
        parquet_path=str(train_path), target_col=cfg.data.target_col,
        filter_cols=filter_cols,
        include_errors=cfg.benchmark.include_mag_errors,
        log_target=cfg.data.log_target,
    )
    test_ds = FlatVectorDataset(
        parquet_path=str(test_path), target_col=cfg.data.target_col,
        filter_cols=filter_cols,
        include_errors=cfg.benchmark.include_mag_errors,
        log_target=cfg.data.log_target,
        norm_stats=train_ds.norm_stats,
    )

    model = build_benchmark_model(cfg, test_ds.n_features)
    model.load_state_dict(torch.load(run_dir / "best_model.pt", map_location="cpu", weights_only=True))
    model.eval()

    is_prob   = cfg.benchmark.model_type == "flat_mdn"
    head_type = "MDN" if is_prob else "MLPRegressor"

    features = torch.from_numpy(test_ds.features)
    z_raw    = torch.from_numpy(test_ds.targets)
    z_real   = np.expm1(z_raw.numpy()) if cfg.data.log_target else z_raw.numpy()

    state = {
        "model":            model,
        "features":         features,
        "z_true":           z_real,
        "z_true_raw":       z_raw,
        "cfg":              cfg,
        "filter_cols":      filter_cols,
        "is_benchmark":     True,
        "is_probabilistic": is_prob,
        "head_type":        head_type,
        "benchmark_subset": cfg.benchmark.subset_name,
        "benchmark_type":   cfg.benchmark.model_type,
    }
    _cache[run_name] = state
    return state


def load_run(run_name: str, role: str | None = None) -> dict:
    run_dir = _resolve_run_dir(run_name)
    if role is None:
        role = default_role(run_dir)
    key = _cache_key(run_name, role)
    if key in _cache:
        return _cache[key]

    cfg = load_config(run_dir / "config.yaml")

    if cfg.benchmark.enabled:
        return _load_benchmark_run(run_name, run_dir, cfg)

    if role == ROLE_POINT:
        head_type = "mlp_regressor"
    elif role in (ROLE_POSTERIOR, ROLE_CALIBRATED):
        head_type = (
            cfg.training.stage2.head if cfg.training.split_training and cfg.training.stage2.head
            else cfg.head.type
        )
    else:
        head_type = cfg.head.type

    ckpt = checkpoint_path(run_dir, role)
    if ckpt is None:
        raise FileNotFoundError(f"No checkpoint for role '{role}' in {run_dir}")

    model = build_model(cfg, head_type=head_type)
    model.load_state_dict(torch.load(ckpt, map_location="cpu", weights_only=True))
    model.eval()

    head      = model.head
    is_prob   = isinstance(head, PROB_HEADS)
    head_type = head.__class__.__name__
    sigma_scale = 1.0
    if role == ROLE_CALIBRATED:
        sigma_scale = post_hoc_sigma_scale(run_dir) or 1.0

    test_path = ROOT / cfg.data.test_path
    res_dir   = ROOT / cfg.data.res_dir
    test_ds   = GalaxyDataset(
        parquet_path   = test_path,
        res_dir        = res_dir,
        target_col     = cfg.data.target_col,
        dropout_cfg    = None,
        active_surveys = cfg.data.active_surveys or None,
        log_target     = cfg.data.log_target,
        include_errors = getattr(cfg.data, "include_errors", False),
    )

    loader = DataLoader(test_ds, batch_size=2048, shuffle=False,
                        collate_fn=collate_fn, num_workers=0)

    all_tokens, all_masks, all_z_raw = [], [], []
    with torch.no_grad():
        for t, m, z in loader:
            all_tokens.append(t)
            all_masks.append(m)
            all_z_raw.append(z)

    z_raw  = torch.cat(all_z_raw)
    z_real = np.expm1(z_raw.numpy()) if cfg.data.log_target else z_raw.numpy()

    _cache[key] = {
        "model":            model,
        "tokens":           torch.cat(all_tokens),
        "masks":            torch.cat(all_masks),
        "z_true":           z_real,
        "z_true_raw":       z_raw,
        "cfg":              cfg,
        "filters":          test_ds.filters,
        "col_index":        {fi.col_name: i for i, fi in enumerate(test_ds.filters)},
        "is_benchmark":     False,
        "is_probabilistic": is_prob,
        "head_type":        head_type,
        "role":             role,
        "sigma_scale":      sigma_scale,
        "run_dir":          run_dir,
        "available_roles":  available_roles(run_dir),
        "post_hoc":         load_post_hoc(run_dir),
    }
    return _cache[key]


def _build_sub_mask(state: dict, selected_cols: list) -> torch.Tensor:
    masks     = state["masks"]
    col_index = state["col_index"]
    sub_mask  = torch.ones_like(masks)
    for col in selected_cols:
        if col in col_index:
            sub_mask[:, col_index[col]] = masks[:, col_index[col]]
    return sub_mask


def _collect_prob_outputs(state: dict, sub_mask: torch.Tensor, cols_key: tuple) -> None:
    """
    Run inference on a probabilistic head and cache mean / median / mode / PIT.
    Results are stored in _prob_cache[(run_name_placeholder, cols_key)].
    Note: call sites use the run_name to build the key.
    """
    model     = state["model"]
    head        = model.head
    head_type   = state["head_type"]
    tokens      = state["tokens"]
    z_raw       = state["z_true_raw"]
    cfg         = state["cfg"]
    sigma_scale = state.get("sigma_scale", 1.0)

    all_pi, all_mu, all_sigma = [], [], []
    all_probs = []
    all_widths, all_heights, all_derivs = [], [], []
    with torch.no_grad():
        for start in range(0, len(tokens), 2048):
            t = tokens[start:start+2048]
            m = sub_mask[start:start+2048]
            out = apply_sigma_scale_to_out(model(t, m), head, sigma_scale)
            if head_type == "MDN":
                all_pi.append(out["pi"].cpu())
                all_mu.append(out["mu"].cpu())
                all_sigma.append(out["sigma"].cpu())
            elif head_type == "BinnedPDF":
                all_probs.append(out["probs"].cpu())
            else:  # NSF
                all_widths.append(out["widths"].cpu())
                all_heights.append(out["heights"].cpu())
                all_derivs.append(out["derivs"].cpu())

    def _to_real(t):
        return np.expm1(t.numpy()) if cfg.data.log_target else t.numpy()

    if head_type == "MDN":
        pi    = torch.cat(all_pi)
        mu    = torch.cat(all_mu)
        sigma = torch.cat(all_sigma)
        ests  = head.point_estimates_from_params(pi, mu, sigma)
        pit   = head.pit_values(pi, mu, sigma, z_raw).numpy()
    elif head_type == "BinnedPDF":
        probs = torch.cat(all_probs)
        ests  = head.point_estimates_from_probs(probs)
        pit   = head.pit_values(probs, z_raw).numpy()
    else:  # NSF
        widths  = torch.cat(all_widths)
        heights = torch.cat(all_heights)
        derivs  = torch.cat(all_derivs)
        ests    = head.point_estimates_from_params(widths, heights, derivs)
        pit     = head.pit_values(widths, heights, derivs, z_raw).numpy()

    return {
        "z_mean":   _to_real(ests["z_mean"]),
        "z_median": _to_real(ests["z_median"]),
        "z_mode":   _to_real(ests["z_mode"]),
        "pit":      pit,
    }


def run_inference(
    run_name: str,
    selected_cols: list,
    estimate: str = "mean",
    role: str | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Run inference with a given filter subset and point estimate type.

    estimate : "mean" | "median" | "mode"
        Ignored for MLP regressor heads (only one estimate available).
        For MDN / BinnedPDF, returns the requested summary statistic.
        Results for all three estimates are cached on first call per filter subset.

    Benchmark runs ignore selected_cols — they use a fixed filter subset.
    """
    run_dir = _resolve_run_dir(run_name)
    if role is None:
        role = default_role(run_dir)
    state   = load_run(run_name, role=role)
    model   = state["model"]
    z_true  = state["z_true"]
    cfg     = state["cfg"]
    is_prob = state["is_probabilistic"]

    # ── Flat benchmark path (fixed features) ─────────────────────────────
    if state.get("is_benchmark"):
        features = state["features"]
        cache_key = (run_name, estimate) if is_prob else (run_name, "mlp")

        if is_prob:
            if cache_key not in _prob_cache:
                head = model.head
                all_pi, all_mu, all_sigma = [], [], []
                with torch.no_grad():
                    for start in range(0, len(features), 2048):
                        x = features[start:start+2048]
                        out = model(x)
                        all_pi.append(out["pi"])
                        all_mu.append(out["mu"])
                        all_sigma.append(out["sigma"])
                pi, mu, sigma = torch.cat(all_pi), torch.cat(all_mu), torch.cat(all_sigma)
                ests = head.point_estimates_from_params(pi, mu, sigma)
                pit  = head.pit_values(pi, mu, sigma, state["z_true_raw"]).numpy()
                def _to_real(t):
                    return np.expm1(t.numpy()) if cfg.data.log_target else t.numpy()
                _prob_cache[cache_key] = {
                    "z_mean": _to_real(ests["z_mean"]),
                    "z_median": _to_real(ests["z_median"]),
                    "z_mode": _to_real(ests["z_mode"]),
                    "pit": pit,
                }
            return z_true, _prob_cache[cache_key][f"z_{estimate}"]

        z_preds = []
        with torch.no_grad():
            for start in range(0, len(features), 2048):
                out = model(features[start:start+2048])
                z_preds.append(out["z_pred"])
        z_pred_np = torch.cat(z_preds).numpy()
        if cfg.data.log_target:
            z_pred_np = np.expm1(z_pred_np)
        return z_true, z_pred_np

    # ── DeepSetZ path (user-selected filter subset) ──────────────────────
    tokens  = state["tokens"]
    masks   = state["masks"]
    sub_mask  = _build_sub_mask(state, selected_cols)
    cols_key  = tuple(sorted(selected_cols))
    cache_key = (run_name, role, cols_key)

    if is_prob:
        if cache_key not in _prob_cache:
            _prob_cache[cache_key] = _collect_prob_outputs(state, sub_mask, cols_key)
        z_pred = _prob_cache[cache_key][f"z_{estimate}"]
        return z_true, z_pred

    z_preds = []
    with torch.no_grad():
        for start in range(0, len(tokens), 2048):
            t = tokens[start:start+2048]
            m = sub_mask[start:start+2048]
            out = model(t, m)
            z_preds.append(out["z_pred"])
    z_pred_np = torch.cat(z_preds).numpy()
    if cfg.data.log_target:
        z_pred_np = np.expm1(z_pred_np)
    return z_true, z_pred_np


# ── Plotting helpers ─────────────────────────────────────────────────────────
def _sigma_nmad(delta_z):
    return 1.4826 * np.median(np.abs(delta_z - np.median(delta_z)))

def make_results_figure(z_true, z_pred, filter_labels: list[str], run_name: str,
                        estimate_label: str = ""):
    delta_z  = (z_pred - z_true) / (1.0 + z_true)
    sigma    = _sigma_nmad(delta_z)
    bias     = float(np.median(delta_z))
    outlier  = float(np.mean(np.abs(delta_z) > OUTLIER_THRESHOLD))
    rmse     = float(np.sqrt(np.mean(delta_z**2)))
    n        = len(z_true)

    filters_str = "  ·  ".join(filter_labels) if filter_labels else "—"
    est_suffix  = f"   [{estimate_label}]" if estimate_label else ""

    fig = plt.figure(figsize=(14, 10))
    fig.patch.set_facecolor("#f8f9fa")

    # Title + metrics strip
    title = (f"{run_name}{est_suffix}   |   {len(filter_labels)} filter(s): {filters_str}\n"
             f"N={n:,}   σ_NMAD={sigma:.4f}   bias={bias:+.4f}   "
             f"outlier={outlier*100:.1f}%   RMSE={rmse:.4f}")
    fig.suptitle(title, fontsize=10, family="monospace",
                 y=0.98, va="top", color="#222")

    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32,
                           top=0.90, bottom=0.08, left=0.07, right=0.97)

    # ── 1. z_phot vs z_spec scatter ─────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    zmax = max(z_true.max(), z_pred.max()) * 1.05
    h = ax1.hexbin(z_true, z_pred, gridsize=70, cmap="YlOrRd",
                   mincnt=1, bins="log")
    fig.colorbar(h, ax=ax1, label="log₁₀(count)", pad=0.02)
    z_line = np.array([0, zmax])
    ax1.plot(z_line, z_line, "k--", lw=1.0, label="1:1")
    ax1.plot(z_line, z_line + OUTLIER_THRESHOLD*(1+z_line), "k:", lw=0.7, alpha=0.5)
    ax1.plot(z_line, z_line - OUTLIER_THRESHOLD*(1+z_line), "k:", lw=0.7, alpha=0.5)
    ax1.set_xlim(0, zmax); ax1.set_ylim(0, zmax)
    ax1.set_xlabel("$z_{\\rm spec}$"); ax1.set_ylabel("$z_{\\rm phot}$")
    ax1.set_title("$z_{\\rm phot}$ vs $z_{\\rm spec}$", fontsize=10)
    ax1.legend(fontsize=8)

    # ── 2. Δz histogram ──────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(delta_z, bins=100, range=(-0.4, 0.4),
             color="#4C8BE2", edgecolor="none", alpha=0.85, density=True)
    ax2.axvline(0,    color="k",      lw=1.0, ls="--")
    ax2.axvline(bias, color="#E2734C", lw=1.5, label=f"bias={bias:+.4f}")
    ax2.axvline( OUTLIER_THRESHOLD, color="gray", lw=0.8, ls=":")
    ax2.axvline(-OUTLIER_THRESHOLD, color="gray", lw=0.8, ls=":")
    ax2.set_xlabel("$\\Delta z = (z_{\\rm phot}-z_{\\rm spec})/(1+z_{\\rm spec})$",
                   fontsize=9)
    ax2.set_ylabel("Density")
    ax2.set_title("$\\Delta z$ distribution", fontsize=10)
    ax2.legend(fontsize=8)
    ax2.grid(True, axis="y", alpha=0.3)

    # ── 3. σ_NMAD and bias vs redshift ───────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, :])
    z_edges = np.linspace(0, z_true.max() * 1.02, 16)
    z_cens  = 0.5 * (z_edges[:-1] + z_edges[1:])
    sigmas, biases_bin, outliers_bin, counts = [], [], [], []
    for lo, hi in zip(z_edges[:-1], z_edges[1:]):
        m = (z_true >= lo) & (z_true < hi)
        counts.append(m.sum())
        if m.sum() > 20:
            dz = delta_z[m]
            sigmas.append(_sigma_nmad(dz))
            biases_bin.append(float(np.median(dz)))
            outliers_bin.append(float(np.mean(np.abs(dz) > OUTLIER_THRESHOLD)))
        else:
            sigmas.append(np.nan); biases_bin.append(np.nan); outliers_bin.append(np.nan)

    color_s = "#4C8BE2"; color_b = "#E2734C"; color_o = "#4CE27A"
    ax3.plot(z_cens, sigmas,      "o-", color=color_s, lw=1.5, ms=5, label="σ_NMAD")
    ax3.plot(z_cens, biases_bin,  "s--", color=color_b, lw=1.2, ms=4, label="bias")
    ax3.plot(z_cens, outliers_bin,"^:", color=color_o, lw=1.2, ms=4, label="outlier rate")
    ax3.axhline(0, color="k", lw=0.7, ls="--", alpha=0.5)
    ax3.set_xlabel("$z_{\\rm spec}$"); ax3.set_ylabel("Metric value")
    ax3.set_title("Performance vs redshift (binned)", fontsize=10)
    ax3.legend(fontsize=9, ncol=3)
    ax3.grid(True, alpha=0.25)

    # Counts bar (twin axis)
    ax3b = ax3.twinx()
    ax3b.bar(z_cens, counts, width=(z_edges[1]-z_edges[0])*0.8,
             color="gray", alpha=0.15, label="N per bin")
    ax3b.set_ylabel("N galaxies", color="gray", fontsize=8)
    ax3b.tick_params(axis="y", colors="gray", labelsize=7)

    ipy_display(fig)
    plt.close(fig)


# ── Calibration figure (MDN / BinnedPDF only) ───────────────────────────────
def _compute_coverage(pit: np.ndarray, alphas: np.ndarray) -> np.ndarray:
    return np.array([
        np.mean((pit >= (1 - a) / 2) & (pit <= (1 + a) / 2))
        for a in alphas
    ])

def make_calibration_figure(
    run_name:      str,
    selected_cols: list,
    filter_labels: list,
    role: str | None = None,
):
    """
    Show PIT histogram + coverage/reliability diagram for a probabilistic model.
    Requires that _prob_cache already has the entry for this selection
    (call run_inference first).
    """
    run_dir = _resolve_run_dir(run_name)
    if role is None:
        role = default_role(run_dir)
    cols_key  = tuple(sorted(selected_cols))
    cache_key = (run_name, role, cols_key)
    if cache_key not in _prob_cache:
        raise RuntimeError("Run inference first before showing calibration.")
    pit_vals = _prob_cache[cache_key]["pit"]
    pit      = np.asarray(pit_vals, dtype=float)
    pit      = pit[np.isfinite(pit)]

    alphas   = np.linspace(0.0, 1.0, 101)
    coverage = _compute_coverage(pit, alphas)

    state     = load_run(run_name, role=role)
    head_type = state["head_type"]
    role_lbl  = role_label(role)
    calib_note = ""
    if state.get("post_hoc") and role == ROLE_CALIBRATED:
        s = state["post_hoc"].get("sigma_scale", 1.0)
        calib_note = f"  |  σ scale={s:.3f}"
    filters_str = "  ·  ".join(filter_labels) if filter_labels else "—"

    fig, (ax_pit, ax_cov) = plt.subplots(1, 2, figsize=(13, 5))
    fig.patch.set_facecolor("#f8f9fa")
    fig.suptitle(
        f"{run_name}  [{head_type} · {role_lbl}{calib_note}]  |  "
        f"{len(filter_labels)} filter(s): {filters_str}",
        fontsize=10, fontweight="bold", y=1.01,
    )

    # PIT histogram
    ax_pit.hist(pit, bins=25, range=(0, 1), density=True,
                color="#4C8BE2", edgecolor="white", linewidth=0.4, alpha=0.85)
    ax_pit.axhline(1.0, color="k", lw=1.5, ls="--", label="Uniform (ideal)")
    ax_pit.set_xlim(0, 1)
    ax_pit.set_xlabel("PIT value", fontsize=11)
    ax_pit.set_ylabel("Density", fontsize=11)
    ax_pit.set_title("PIT Histogram", fontsize=11)
    ax_pit.legend(fontsize=9)
    ax_pit.grid(True, axis="y", alpha=0.3)
    ks_stat = float(np.max(np.abs(np.sort(pit) - np.linspace(0, 1, len(pit)))))
    ax_pit.text(0.97, 0.97, f"KS={ks_stat:.3f}\nN={len(pit):,}",
                transform=ax_pit.transAxes, ha="right", va="top",
                fontsize=9, family="monospace",
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))

    # Coverage / reliability diagram
    ax_cov.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration", zorder=3)
    ax_cov.fill_between(alphas, alphas, coverage,
                        where=(coverage > alphas),
                        alpha=0.25, color="#4CE27A", label="Over-dispersed")
    ax_cov.fill_between(alphas, alphas, coverage,
                        where=(coverage < alphas),
                        alpha=0.25, color="#E2734C", label="Under-dispersed")
    ax_cov.plot(alphas, coverage, color="#4C8BE2", lw=2.0, label="Model", zorder=4)
    ax_cov.set_xlim(0, 1); ax_cov.set_ylim(0, 1)
    ax_cov.set_xlabel("Confidence level α", fontsize=11)
    ax_cov.set_ylabel("Coverage fraction", fontsize=11)
    ax_cov.set_title("Coverage / Reliability Diagram", fontsize=11)
    ax_cov.legend(fontsize=9, loc="upper left")
    ax_cov.grid(True, alpha=0.3)
    mace = float(np.mean(np.abs(coverage - alphas)))
    ax_cov.text(0.97, 0.03, f"MACE={mace:.4f}",
                transform=ax_cov.transAxes, ha="right", va="bottom",
                fontsize=9, family="monospace",
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none"))

    plt.tight_layout()
    ipy_display(fig)
    plt.close(fig)


print("✓ Backend loaded. Run the cell below to launch the UI.")


✓ Backend loaded. Run the cell below to launch the UI.


In [ ]:
# ── Interactive UI — run this cell to launch ────────────────────────────────
import ipywidgets as widgets
from IPython.display import display

# ── Discover available runs ──────────────────────────────────────────────────
runs = find_runs(OUTPUTS_DIR, BENCHMARKS_DIR)
if not runs:
    print("No completed runs found in outputs/ or benchmarks/. Train a model first.")
else:

    # ── Widgets ───────────────────────────────────────────────────────────────
    w_model = widgets.Dropdown(
        options    = list(runs.keys()),
        description= "",
        layout     = widgets.Layout(width="320px"),
        style      = {"description_width": "0px"},
    )

    w_role = widgets.Dropdown(
        options    = [],
        description= "View:",
        layout     = widgets.Layout(width="280px"),
        style      = {"description_width": "42px"},
    )

    w_status = widgets.HTML(value="<i style='color:#888'>Select a model and filters, then press Run.</i>")

    # ── Point-estimate selector (MDN / BinnedPDF only) ────────────────────────
    w_estimate = widgets.ToggleButtons(
        options    = [("Mean", "mean"), ("Median", "median"), ("Mode", "mode")],
        value      = "mean",
        description= "Estimate:",
        layout     = widgets.Layout(width="auto"),
        style      = {"button_width": "72px", "description_width": "62px"},
        disabled   = True,   # enabled once a probabilistic model is loaded
    )
    estimate_label_map = {"mean": "Mean", "median": "Median", "mode": "Mode"}

    # Filter checkboxes — built dynamically when model changes
    filter_boxes    = {}   # col_name -> Checkbox
    survey_sections = {}   # survey -> VBox

    def _make_checkbox(col, disabled=False):
        label = FILTER_LABELS.get(col, col)
        cb = widgets.Checkbox(
            value       = True,
            description = label,
            disabled    = disabled,
            layout      = widgets.Layout(width="110px"),
            style       = {"description_width": "0px"},
            indent      = False,
        )
        return cb

    def _survey_header(title, survey_key):
        btn_all  = widgets.Button(description="All",  layout=widgets.Layout(width="48px", height="24px"),
                                  button_style="")
        btn_none = widgets.Button(description="None", layout=widgets.Layout(width="52px", height="24px"),
                                  button_style="")
        def _set_all(b):
            for col in SURVEY_DISPLAY.get(survey_key, []):
                if col in filter_boxes and not filter_boxes[col].disabled:
                    filter_boxes[col].value = True
        def _set_none(b):
            for col in SURVEY_DISPLAY.get(survey_key, []):
                if col in filter_boxes and not filter_boxes[col].disabled:
                    filter_boxes[col].value = False
        btn_all.on_click(_set_all)
        btn_none.on_click(_set_none)
        label = widgets.HTML(f"<b style='font-size:13px'>{title}</b>")
        return widgets.HBox([label, btn_all, btn_none],
                            layout=widgets.Layout(align_items="center", gap="6px"))

    filter_panel = widgets.VBox([], layout=widgets.Layout(
        border="1px solid #ddd", padding="10px 14px",
        border_radius="6px", background_color="#fafafa",
    ))

    def _refresh_role_options(run_name: str, select: str | None = None):
        run_dir = _resolve_run_dir(run_name)
        roles = available_roles(run_dir)
        if not roles:
            roles = [ROLE_END_TO_END]
        w_role.options = [(role_label(r), r) for r in roles]
        pick = select or default_role(run_dir)
        if pick not in roles:
            pick = roles[0]
        w_role.value = pick

    def rebuild_filters(run_name, role: str | None = None):
        filter_boxes.clear()
        if role is None:
            role = w_role.value if w_role.options else default_role(_resolve_run_dir(run_name))
        state = load_run(run_name, role=role)
        is_bench = state.get("is_benchmark", False)

        if is_bench:
            # Benchmark: fixed filter subset — show read-only checkboxes
            fixed_cols = state["filter_cols"]
            sections = []
            for survey_key in SURVEY_ORDER:
                cols_in_survey = [c for c in SURVEY_DISPLAY.get(survey_key, [])
                                  if c in fixed_cols]
                if not cols_in_survey:
                    continue
                header = widgets.HTML(
                    f"<b style='font-size:13px'>{SURVEY_TITLES[survey_key]}</b>"
                    f" <span style='color:#888;font-size:11px'>(fixed)</span>"
                )
                row_widgets = []
                for col in cols_in_survey:
                    cb = _make_checkbox(col, disabled=True)
                    cb.value = True
                    filter_boxes[col] = cb
                    row_widgets.append(cb)
                row = widgets.HBox(row_widgets, layout=widgets.Layout(flex_wrap="wrap"))
                sections.append(widgets.VBox([header, row],
                                layout=widgets.Layout(margin="4px 0 8px 0")))
            filter_panel.children = sections
            btn_all_global.disabled = True
            btn_none_global.disabled = True
        else:
            btn_all_global.disabled = False
            btn_none_global.disabled = False
            active_cols = {fi.col_name for fi in state["filters"]}
            sections = []
            for survey_key in SURVEY_ORDER:
                cols_in_survey = [c for c in SURVEY_DISPLAY.get(survey_key, [])
                                  if c in active_cols]
                if not cols_in_survey:
                    continue
                header = _survey_header(SURVEY_TITLES[survey_key], survey_key)
                row_widgets = []
                for col in cols_in_survey:
                    cb = _make_checkbox(col)
                    filter_boxes[col] = cb
                    row_widgets.append(cb)
                row = widgets.HBox(row_widgets, layout=widgets.Layout(flex_wrap="wrap"))
                sections.append(widgets.VBox([header, row],
                                layout=widgets.Layout(margin="4px 0 8px 0")))
            filter_panel.children = sections

        cfg       = state["cfg"]
        is_prob   = state["is_probabilistic"]
        n_filters = len(state.get("filter_cols", [fi.col_name for fi in state.get("filters", [])]))
        w_estimate.disabled = not is_prob
        btn_calib.disabled  = True
        head_info = f"  [{role_label(state.get('role', ''))}]"
        if state.get("post_hoc") and state.get("role") == ROLE_CALIBRATED:
            head_info += f" σ×{state['post_hoc']['sigma_scale']:.3f}"
        bench_info = (
            f"  <span style='color:#888'>benchmark · {state['benchmark_subset']}</span>"
            if is_bench else ""
        )
        w_status.value = (
            f"<span style='color:#555'>Loaded <b>{run_name}</b>{head_info}{bench_info} — "
            f"{n_filters} filter(s), "
            f"{'log(1+z) target' if cfg.data.log_target else 'linear z target'}</span>"
        )

    # Global select / deselect all
    btn_all_global  = widgets.Button(description="Select all",   button_style="info",
                                     layout=widgets.Layout(width="100px"))
    btn_none_global = widgets.Button(description="Deselect all", button_style="warning",
                                     layout=widgets.Layout(width="110px"))
    btn_run         = widgets.Button(description="▶  Run",       button_style="success",
                                     layout=widgets.Layout(width="100px"))
    btn_calib       = widgets.Button(description="📊 Calibration", button_style="info",
                                     layout=widgets.Layout(width="130px"),
                                     disabled=True,
                                     tooltip="Show PIT histogram and coverage diagram")

    def _all_global(b):
        for cb in filter_boxes.values():
            if not cb.disabled: cb.value = True
    def _none_global(b):
        for cb in filter_boxes.values():
            if not cb.disabled: cb.value = False
    btn_all_global.on_click(_all_global)
    btn_none_global.on_click(_none_global)

    output = widgets.Output()

    def on_run(b):
        run_name = w_model.value
        role     = w_role.value
        state    = load_run(run_name, role=role)
        is_bench = state.get("is_benchmark", False)
        if is_bench:
            selected = list(state.get("filter_cols", []))
        else:
            selected = [col for col, cb in filter_boxes.items() if cb.value]
        if len(selected) == 0:
            w_status.value = "<span style='color:#c00'><b>Please select at least one filter.</b></span>"
            return

        labels    = [FILTER_LABELS.get(c, c) for c in selected]
        estimate  = w_estimate.value                          # "mean" / "median" / "mode"
        est_label = estimate_label_map[estimate]
        is_prob   = state.get("is_probabilistic", False)
        est_display = f" [{est_label}]" if is_prob else ""

        w_status.value = (
            f"<span style='color:#555'>Running inference{est_display} on "
            f"{len(selected)} filter(s)…</span>"
        )
        btn_run.disabled = True
        output.clear_output(wait=True)
        try:
            z_true, z_pred = run_inference(run_name, selected, estimate=estimate, role=role)
            with output:
                make_results_figure(z_true, z_pred, labels, run_name,
                                    estimate_label=est_label if is_prob else "")
            if is_prob:
                btn_calib.disabled = False
            w_status.value = (
                f"<span style='color:#2a7'>✓ Done — "
                f"{len(selected)} filter(s): {', '.join(labels)}{est_display}</span>"
            )
        except Exception as e:
            w_status.value = f"<span style='color:#c00'>Error: {e}</span>"
        finally:
            btn_run.disabled = False

    def on_calib(b):
        run_name = w_model.value
        role     = w_role.value
        selected = [col for col, cb in filter_boxes.items() if cb.value]
        if len(selected) == 0:
            w_status.value = "<span style='color:#c00'><b>Please select at least one filter.</b></span>"
            return
        labels = [FILTER_LABELS.get(c, c) for c in selected]
        output.clear_output(wait=True)
        try:
            with output:
                make_calibration_figure(run_name, selected, labels, role=role)
            w_status.value = (
                f"<span style='color:#2a7'>✓ Calibration — "
                f"{len(selected)} filter(s): {', '.join(labels)}</span>"
            )
        except Exception as e:
            w_status.value = f"<span style='color:#c00'>Error: {e}</span>"

    btn_calib.on_click(on_calib)

    btn_run.on_click(on_run)

    def on_model_change(change):
        if change["name"] == "value":
            w_status.value = "<span style='color:#888'>Loading model…</span>"
            output.clear_output()
            try:
                _refresh_role_options(change["new"])
                rebuild_filters(change["new"], role=w_role.value)
            except Exception as e:
                w_status.value = f"<span style='color:#c00'>Failed to load model: {e}</span>"

    def on_role_change(change):
        if change["name"] == "value" and w_model.value:
            _prob_cache.clear()
            w_status.value = "<span style='color:#888'>Switching model view…</span>"
            output.clear_output()
            try:
                rebuild_filters(w_model.value, role=change["new"])
            except Exception as e:
                w_status.value = f"<span style='color:#c00'>Failed to load role: {e}</span>"

    w_model.observe(on_model_change, names="value")
    w_role.observe(on_role_change, names="value")

    # ── Layout ────────────────────────────────────────────────────────────────
    header = widgets.HTML(
        "<h3 style='margin:0 0 4px 0; font-family:sans-serif'>DeepSetZ — Interactive Evaluation</h3>"
    )
    model_row = widgets.HBox(
        [widgets.HTML("<b>Model:</b>"), w_model, w_role],
        layout=widgets.Layout(align_items="center", gap="8px", margin="4px 0"),
    )
    btn_row = widgets.HBox(
        [btn_all_global, btn_none_global,
         widgets.HTML("<span style='flex:1'></span>"),
         w_estimate, btn_run, btn_calib],
        layout=widgets.Layout(align_items="center", gap="8px", margin="6px 0 0 0"),
    )
    ui = widgets.VBox(
        [header, model_row, filter_panel, btn_row, w_status, output],
        layout=widgets.Layout(max_width="960px", padding="10px"),
    )

    # Trigger initial load
    _refresh_role_options(w_model.value)
    rebuild_filters(w_model.value, role=w_role.value)
    display(ui)
